In [ ]:
%pip install -e ../../../
%pip install plotly
%pip install nbformat

In [ ]:
import itertools

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from water_futures_battle.core import Settings
from water_futures_battle.core.utility import timestampify
from water_futures_battle.core.views import get_snapshot, YearlyView

from water_futures_battle.climate import configure_climate
from water_futures_battle.economy import configure_economy
from water_futures_battle.nrw_model import configure_nrw_model, NRWClass, NRWModelSettings
from water_futures_battle.nrw_model.dynamic_properties import NRWModelDB
from water_futures_battle.nrw_model.policies import NRWMitigation
from water_futures_battle.water_demand_model import configure_water_demand_model

from water_futures_battle.jurisdictions import (
    build_state,
    Municipality,
    MunicipalitySize,
    generate_nrw_demand,
    generate_water_demand,
    age_distribution_networks,
)

from water_futures_battle.masterplan.default_historical_data import get_historical_masterplan
from water_futures_battle.masterplan.entities import Masterplan

# Directly define the config dictionary (from configuration.yaml)
config = {
    'state': {
        'name': 'Netherlands',
        'id': 'NL0000',
        'jurisdictions-static_properties': './jurisdictions/jurisdictions-static_properties.xlsx',
        'municipalities-dynamic_properties': './jurisdictions/municipalities-dynamic_properties.xlsx',
    },
    'water_demand_model': {
        'water_demand_model-static_properties': './water_demand_model/water_demand_model-static_properties.xlsx',
        'water_demand_model-dynamic_properties': './water_demand_model/water_demand_model-dynamic_properties.xlsx',
    },
    'nrw_model': {
        'nrw_model-intervention_success_prob-min': 0.9,
        'nrw_model-dynamic_properties': './nrw_model/nrw_model-dynamic_properties.xlsx',
    },
    'climate': {
        'climate-dynamic_properties': './climate/climate-dynamic_properties.xlsx',
    },
    'economy': {
        'economy-dynamic_properties': './economy/economy-dynamic_properties.xlsx',
        'bonds': {
            'risk_free_rate': 1.0,
            'spread_sensitivity': 1.0,
            'maturity': 25,
            'static_properties': './economy/bonds-static_properties.xlsx',
        },
    },
    'settings': {
        'start_year': 2000,
        'end_year': 2024,
        'national_budget': 0,
        'lifeline_volume': 80,
        'seed': 128
    },
    'data_path': '../../'
}

masterplan = Masterplan(get_historical_masterplan())

settings = Settings.from_config(config[Settings.LABEL])

climate_db = configure_climate(
    config['climate'],
    config["data_path"],
    settings
    )

bonds_settings, economy_db, utilities_bonds = configure_economy(
    config['economy'],
    config["data_path"],
    settings
    )

state = build_state(
    config['state'],
    config["data_path"],
    settings
    )

water_dem_patterns, water_dem_db = configure_water_demand_model(
    config['water_demand_model'],
    config["data_path"],
    settings
    )
nrw_settings, nrw_db = configure_nrw_model(
    config['nrw_model'],
    config["data_path"],
    settings
    )

wu_pvs = [
    ["PV20"],
    ["PV21"],
    ["PV22"],
    ["PV23"],
    ["PV24", "PV25"],
    ["PV26"],
    ["PV27"],
    ["PV28", "PV29"],
    ["PV30"],
    ["PV31"]
]
def wu_id_gen(x: int) -> str:
    return f"WU{x:02d}"
water_utilities_map = {
    wu_id_gen(i+1): wu_pv for i, wu_pv in enumerate(wu_pvs)
}

def apply_nrw_interventions(
        utility_id: str,
        municipalities: set[Municipality],
        year: int,
        policy_desc: dict,
        settings: Settings,
        nrw_settings: NRWModelSettings,
        nrw_info_db: NRWModelDB
    ) -> float:

    munis = [get_snapshot(m, year) for m in sorted(municipalities, key=lambda x: x.cbs_id)]

    budget = policy_desc['budget']

    # Let's distribute the budget based on the policy
    if policy_desc['policy'] == NRWMitigation.BY_NRW_CLASS:
        budget_allocation_plan = NRWMitigation.distribute_budget_by_nrw_class(
            budget,
            munis,
            year,
            nrw_info_db
        )
    elif policy_desc['policy'] == NRWMitigation.BY_POPULATION:
        budget_allocation_plan = NRWMitigation.distribute_budget_by_population(
            budget,
            munis
        )
    elif policy_desc['policy'] == NRWMitigation.CUSTOM:
        budget_allocation_plan = NRWMitigation.distribute_budget_custom(
            budget,
            munis,
            policy_desc['policy_args']
        )
    else:
        raise ValueError(f"Unknown policy for NRW mitigation: {policy_desc['policy']}")

    # However, not every year we get the same success, let's get the random generator
    RNG = settings.get_random_generator('nrw_model-intervention_succes_prob')

    success_probabilities = RNG.uniform(
            low=nrw_settings.success_proba_bounds[0],
            high=nrw_settings.success_proba_bounds[1],
            size=len(NRWClass) * len(MunicipalitySize)
    )
    success_probabilities_map = { 
        (nrw_class, muni_size_class): success_probabilities[i]
        for i, (nrw_class, muni_size_class) in enumerate(itertools.product(NRWClass, MunicipalitySize))
    }

    # Spend budget on the municipalities
    def budget_to_years(a_budget: float, a_municipality: YearlyView[Municipality]) -> float:
        
        # unit cost is in €/km/year
        unit_cost = nrw_info_db[NRWModelDB.COST].loc[
            timestampify(year),
            f'{a_municipality.state.cbs_id}-{a_municipality.nrw_class.name}-{a_municipality.size_class.name}'
        ]

        # A perfect investement with this budget in this city would give me 
        ideal_years = a_budget / (unit_cost*a_municipality.dist_network_length)

        # However not all investments are equally successfull, 
        success_proba = success_probabilities_map[(a_municipality.nrw_class, a_municipality.size_class)]

        years_dec = ideal_years * success_proba
        if a_municipality.dist_network_avg_age - years_dec <= 0:
            return a_municipality.dist_network_avg_age
        else:
            return years_dec
        
    budget_spent = 0
    for muni in munis:
        if muni not in budget_allocation_plan or budget_allocation_plan[muni] == 0:
            continue

        budget_muni = budget_allocation_plan[muni]
        
        if budget_spent + budget_muni > budget:
            budget_muni = budget - budget_spent

        budget_spent += budget_muni
        muni.update_dist_net_age(when=year, by=-1 * budget_to_years(budget_muni, muni))

    return budget_spent


household_series = {}
business_series = {}
nrw_series = {}

# Step 2: start the evaluation

for year in settings.years_to_simulate:
    print(year)
    mean_tempmax = climate_db['temperature-max-avg'][state.cbs_id]
    maxtemp_year = mean_tempmax.loc[mean_tempmax.index.year == year].max()

    nat_policies = masterplan.national_policies(year=year)
    nat_interven = masterplan.national_interventions(year=year)
    
    for water_utility_id, water_utility_pvs in water_utilities_map.items():
        print(water_utility_id, end=' | ')
        
        municipalities: set[Municipality] = set()
        for pv_id in water_utility_pvs:
            province = state.province(pv_id)
            municipalities = municipalities | province.active_municipalities(when=year)

        # Apply this year interventions (add components, set decisions)
        utility_budget = 0

        wutil_policies = masterplan.water_utility_policies(water_utility=water_utility_id, year=year)
        wutil_interven = masterplan.water_utility_interventions(water_utility=water_utility_id, year=year)

        nrw_actions_cost = apply_nrw_interventions(
            utility_id=water_utility_id,
            municipalities=municipalities,
            year=year,
            policy_desc=wutil_policies['nrw_mitigation'],
            settings=settings,
            nrw_settings=nrw_settings,
            nrw_info_db=nrw_db
        )
        
        utility_budget -= nrw_actions_cost

    for water_utility_id, water_utility_pvs in water_utilities_map.items(): 
        municipalities: set[Municipality] = set()
        for pv_id in water_utility_pvs:
            province = state.province(pv_id)
            municipalities = municipalities | province.active_municipalities(when=year)

        for municipality in sorted(municipalities, key=lambda x: x.cbs_id):
            household_demand, business_demand = generate_water_demand(
                (water_dem_patterns, water_dem_db),
                municipality,
                year,
                max_yearly_temperature=maxtemp_year,
                settings=settings
            )
            bill_demand = household_demand + business_demand

            # Create bill_idx: hourly index for the year, length matches household_demand
            bill_idx = pd.date_range(
                start=pd.Timestamp(year=year, month=1, day=1, hour=0),
                periods=len(household_demand),
                freq='h'
            )

            nrw_demand = generate_nrw_demand(
                municipality,
                year,
                nrw_info_db=nrw_db,
                water_demand=bill_demand,
                settings=settings
            )

            # Create nrw_idx: daily index for the year, length matches nrw_demand
            nrw_idx = pd.date_range(
                start=pd.Timestamp(year=year, month=1, day=1),
                periods=len(nrw_demand),
                freq='D'
            )

            # total_demand = bill_demand + np.repeat(nrw_demand, 24, axis=0)

            # Logging part:
            m = municipality.cbs_id
            if m not in household_series:
                household_series[m] = pd.Series(household_demand, index=bill_idx)
                business_series[m] = pd.Series(business_demand, index=bill_idx)
                nrw_series[m] = pd.Series(nrw_demand, index=nrw_idx)
            else:
                household_series[m] = pd.concat([
                    household_series[m],
                    pd.Series(household_demand, index=bill_idx)
                ])
                business_series[m] = pd.concat([
                    business_series[m],
                    pd.Series(business_demand, index=bill_idx)
                ])
                nrw_series[m] = pd.concat([
                    nrw_series[m],
                    pd.Series(nrw_demand, index=nrw_idx)
                ])

            # end municipalities for loop

        # end water utilities for loop
        
    # Update endogenous dynamic properties (inflation, electricity market, components costs)
    # for next year

    # escalate costs
    this_year = timestampify(year)
    next_year = timestampify(year+1)
    inflation_rate = economy_db['inflation'][state.cbs_id].loc[next_year]/100
    
    nrw_base_costs = nrw_db[NRWModelDB.COST].loc[this_year]
    nrw_infl_costs = (1+inflation_rate) * nrw_base_costs
    nrw_db[NRWModelDB.COST].loc[next_year] = nrw_infl_costs
    
    # Age water utilities
    for water_utility_id, water_utility_pvs in water_utilities_map.items():

        munis_y = set()
        munis_Ny = set()
        for pv_id in water_utility_pvs:
            province = state.province(pv_id)
            munis_y = munis_y | province.active_municipalities(when=year)
            munis_Ny = munis_Ny | province.active_municipalities(when=year+1)
        
        non_closing_munis = munis_y & munis_Ny
        closing_munis = munis_y - non_closing_munis

        #------ Inner distribution network age.
        new_munis = age_distribution_networks(closing_munis, non_closing_munis, year, nrw_db)

    print("")
    # end year for loop
# end of the stage loop

In [ ]:
# Combine into DataFrames
house_df = pd.DataFrame(household_series)
busin_df = pd.DataFrame(business_series)
nrw_df = pd.DataFrame(nrw_series)
total_df = house_df + busin_df + nrw_df.resample('h').ffill()

# Add the national demand
house_df['NL0000'] = house_df.sum(axis=1, skipna=True)
busin_df['NL0000'] = busin_df.sum(axis=1, skipna=True)
nrw_df['NL0000'] = nrw_df.sum(axis=1, skipna=True)
total_df['NL0000'] = total_df.sum(axis=1, skipna=True)

# Add the provincial demand and get out the interesting statiscs:
# 1. 24 hours with the highest demand every year by province
# 2. hour with the highest demand every year by municipality
# Note: we sort them to have always the same order (helps with storing this to git too)
provinces_max_daily_demands = {}
municipalities_max_hourly_demands = []
for pv in sorted(state.provinces, key=lambda x: x.cbs_id):
    municipalities_in_pv = [m.cbs_id for m in pv.municipalities if m.cbs_id in house_df.columns]
    municipalities_in_pv = sorted(municipalities_in_pv)
    
    house_df[pv.cbs_id] = house_df[municipalities_in_pv].sum(axis=1, skipna=True)
    busin_df[pv.cbs_id] = busin_df[municipalities_in_pv].sum(axis=1, skipna=True)
    nrw_df[pv.cbs_id] = nrw_df[municipalities_in_pv].sum(axis=1, skipna=True)
    total_df[pv.cbs_id] = total_df[municipalities_in_pv].sum(axis=1, skipna=True)

    # Take the 24 consecutive hours with the highest cumulative demand
    total_cmd =total_df[pv.cbs_id].rolling(window=24).sum()

    provinces_max_daily_demands[pv.cbs_id] = round(total_cmd.groupby(total_cmd.index.year).max(), 2)

    # Take the timestamp with the highest total provincial demand but extract the 
    # demand by municipality
    max_hours = total_df[pv.cbs_id].groupby(total_df.index.year).idxmax()
    mcs_max_h_dem_df = round(total_df.loc[max_hours, municipalities_in_pv].copy(), 2)
    # But let's also track the demand by province to simplify
    mcs_max_h_dem_df[pv.cbs_id] = round(total_df.loc[max_hours, pv.cbs_id], 2)

    # Add also the time of the year because it may be helpful later
    mcs_max_h_dem_df[f'{pv.cbs_id}-time'] = mcs_max_h_dem_df.index.copy()
    mcs_max_h_dem_df.index = mcs_max_h_dem_df.index.year
    
    municipalities_max_hourly_demands.append(mcs_max_h_dem_df)

# Concatenate and save
pvs_max_daily_dem_df = pd.concat(provinces_max_daily_demands, axis=1)
pvs_max_daily_dem_df.index.name = 'year'
pvs_max_daily_dem_df.to_csv('../provinces-max_daily_demands-CMD.csv')

mcs_max_hourly_dem_df = pd.concat(municipalities_max_hourly_demands, axis=1)
mcs_max_hourly_dem_df.index.name = 'year'
mcs_max_hourly_dem_df.to_csv('../jurisdictions-max_hourly_demands-CMH.csv')

In [ ]:
# Define line and fill color
line_color = '#3059D1'
fill_color = 'rgba(48, 89, 209, 0.2)'

def plot_demand_components(house_dem, busin_dem, nrw_dem):
    fig = go.Figure()
    for col in house_dem.columns:
        fig.add_trace(go.Scatter(
            x=house_dem.index,
            y=house_dem[col],
            mode='lines',
            name=f'household_demand {col}',
            line=dict(color=line_color),
            hoverinfo='x+y+name'
        ))
        fig.add_trace(go.Scatter(
            x=busin_dem.index,
            y=busin_dem[col],
            mode='lines',
            name=f'business_demand {col}',
            line=dict(color=line_color),
            hoverinfo='x+y+name'
        ))
        fig.add_trace(go.Scatter(
            x=nrw_dem.index,
            y=nrw_dem[col],
            mode='lines',
            name=f'nrw_demand {col}',
            line=dict(color='#009344'),
            hoverinfo='x+y+name'
        ))
    fig.update_layout(title=f'Demand components', xaxis_title='Time', yaxis_title='Demand [m^3/h]')
    fig.show()

def plot_vewin_figures(house_nl: pd.Series, busin_nl: pd.Series, nrw_nl: pd.Series) -> None:
    house_yearly = house_nl.fillna(0).groupby(house_df.index.year).sum()
    busin_yearly = busin_nl.fillna(0).groupby(busin_df.index.year).sum()
    nrw_yearly = nrw_nl.fillna(0).groupby(nrw_df.index.year).sum() * 24
    total_yearly = house_yearly + busin_yearly + nrw_yearly

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Reproducing Figure 3.8: Household Sales',
            'Reproducing Figure 3.10: NRW',
            'Reproducing Figure 3.9: Business market Sales',
            'Reproducing Figure 3.7: Drinking Water use'
        )
    )
    
    # Top left: Household
    fig.add_trace(
        go.Scatter(
            x=house_yearly.index,
            y=house_yearly.values,
            mode='lines',
            name='household_demand NL0000',
            line=dict(color=line_color, width=2),
            fill='tozeroy',
            fillcolor=fill_color,
            hoverinfo='x+y+name'
        ),
        row=1, col=1
    )
    # Top right: NRW
    fig.add_trace(
        go.Scatter(
            x=nrw_yearly.index,
            y=nrw_yearly.values,
            mode='lines',
            name='nrw_demand NL0000',
            line=dict(color=line_color, width=2),
            fill='tozeroy',
            fillcolor=fill_color,
            hoverinfo='x+y+name'
        ),
        row=1, col=2
    )
    # Bottom left: Business
    fig.add_trace(
        go.Scatter(
            x=busin_yearly.index,
            y=busin_yearly.values,
            mode='lines',
            name='business_demand NL0000',
            line=dict(color=line_color, width=2),
            fill='tozeroy',
            fillcolor=fill_color,
            hoverinfo='x+y+name'
        ),
        row=2, col=1
    )
    # Bottom right: Total
    fig.add_trace(
        go.Scatter(
            x=total_yearly.index,
            y=total_yearly.values,
            mode='lines',
            name='total_demand NL0000',
            line=dict(color=line_color, width=2),
            fill='tozeroy',
            fillcolor=fill_color,
            hoverinfo='x+y+name'
        ),
        row=2, col=2
    )
    fig.update_layout(
        height=800,
        width=1200,
        title_text='National Yearly Demand',
        showlegend=False
    )
    fig.update_xaxes(title_text='Year', row=1, col=1)
    fig.update_xaxes(title_text='Year', row=1, col=2)
    fig.update_xaxes(title_text='Year', row=2, col=1)
    fig.update_xaxes(title_text='Year', row=2, col=2)
    fig.update_yaxes(title_text='Demand [m³/year]', range=[600e6, 900e6], row=1, col=1)
    fig.update_yaxes(title_text='Demand [m³/year]', range=[0, 100e6], row=1, col=2)
    fig.update_yaxes(title_text='Demand [m³/year]', range=[200e6, 400e6], row=2, col=1)
    fig.update_yaxes(title_text='Demand [m³/year]', range=[600e6, 1300e6], row=2, col=2)
    fig.show()

In [ ]:
# Select columns based on args or sum all
VIEW_COLUMNS = []

if len(VIEW_COLUMNS) == 0:
    VIEW_COLUMNS = ['NL0000']

plot_demand_components(house_df[VIEW_COLUMNS], busin_df[VIEW_COLUMNS], nrw_df[VIEW_COLUMNS])

plot_vewin_figures(house_df['NL0000'], busin_df['NL0000'], nrw_df['NL0000'])